# Part 9: Paper Reproduction Workflow

把论文方法转成可审计实验：主张拆解、环境锁定、数据划分、baseline、消融、重复实验与结论边界。

**先修要求**：完成前序章节，了解 NumPy、张量 shape 与训练/验证划分。

## 学习目标与路线

1. 明确问题与输入输出
2. 理解核心结构
3. 运行最小实验
4. 检查指标和 shape
5. 完成练习与遥感迁移

`claim → protocol → baseline → implementation → sanity checks → repeated runs → ablation → report`

In [ ]:
from pathlib import Path
import json
ROOT = Path.cwd().resolve()
while not (ROOT / 'scripts').exists() and ROOT.parent != ROOT: ROOT = ROOT.parent
print('repository:', ROOT)

## 核心概念

- **Claim matrix**：把论文的每个核心主张映射到数据、指标、对照和证据。
- **Reproducibility**：固定代码版本、依赖、随机种子、硬件、数据 checksum 与划分。
- **Ablation**：一次只改变一个因素，并保留强 baseline 与等预算比较。
- **Uncertainty**：至少多随机种子报告均值±标准差，避免把偶然结果写成结论。

## 本地真实训练结果

长训练由脚本执行；这里读取本机已经生成的指标和产物。GitHub 中不会包含 `outputs/`。

In [ ]:
output = ROOT / r'outputs/part9_reproduction'
metrics_path = output / 'metrics.json'
print('local output:', output)
print(json.dumps(json.loads(metrics_path.read_text(encoding='utf-8')), indent=2) if metrics_path.exists() else '尚未运行：请执行下方命令。')

## 可恢复的脚本训练

默认不在 Notebook 内启动长训练。将 `RUN_SHORT_TRAIN` 改为 `True` 可执行一次小规模 smoke run；正式命令请在终端执行。

In [ ]:
RUN_SHORT_TRAIN = False
command = r'''conda run --no-capture-output -n remote-sensing-dl python powershell -ExecutionPolicy Bypass -File scripts/real_training/run_part9.ps1'''
print(command)
if RUN_SHORT_TRAIN:
    import subprocess
    subprocess.run(command, cwd=ROOT, shell=True, check=True)

## 结果解释

正式结果必须包含非空 checkpoint、完整 epoch 日志、验证指标与预测图；出现缺失时先检查数据配对、CUDA 与配置，而不要直接扩大模型。

## 练习

选一篇目标论文，写三行 claim matrix：主张、必要对照、指标、失败时如何解释。

In [ ]:
# 答案脚手架
experiment = {'hypothesis': '', 'controlled_variables': [], 'metric': '', 'expected_failure': ''}
print(experiment)

## 常见坑与扩展

**常见坑**：只跑一个 seed；复现时悄悄改变数据划分；弱 baseline；未记录失败实验；把未复现等同于论文错误。

**扩展**：替换为真实遥感数据时，记录传感器、波段、分辨率、空间划分和随机种子。

## 完成检查

- [ ] 所有单元格从头运行
- [ ] 指标实现通过 sanity check
- [ ] 没有 train/val 空间泄漏
- [ ] 配置和随机种子已记录
- [ ] 至少分析一个失败案例